# Task 1. Repository Cloning and File Discovery 

**Mục tiêu**: Clone repository được phân công và liệt kê/đếm toàn bộ file `Python` , làm input cho pipeline `CPG` ở các task sau.

**Repo được phân công:** [huggingface/transformers-pr-agent](https://github.com/huggingface/transformers-pr-agent)


## Cách làm và Lý do

- **Shallow clone** `--depth 1`: chỉ lấy snapshot mới nhất nên sẽ giảm dung lượng tải (phù hợp với yêu cầu: `reduce download size`).
- Liệt kê bằng `pathlib.rglob("*.py")` thuần `Python`: tái dùng được ở Task 2 (parser cũng cần duyệt file).
- **Quy tắc lọc** `test/setup/generated` + lý do chọn **rule đơn giản** (tránh loại nhầm source thật như `testing_utils.py`...).

### Sơ đồ Quy trình File Discovery

![Task 1 Repository Cloning and File Discovery Flowchart](images/01_file_discovery_flow.png)



### 1. Clone repository:
Chạy 1 lần ở terminal (không chạy lại khi build book):

```bash
git clone --depth 1 https://github.com/huggingface/transformers-pr-agent.git target-repo
```


### 2. Discovery

In [10]:
from pathlib import Path

REPO = Path("../target-repo")

# Count all files python (ignore .git and use recursive)
all_py = [p for p in REPO.rglob("*.py") if ".git" not in p.parts]

def is_excluded(p: Path) -> bool:
    name = p.name
    if name.startswith("test_") or name.endswith("_test.py"):
        return True
    if any(part in ("tests", "test") for part in p.parts):
        return True
    if name in ("setup.py", "conftest.py"):
        return True
    if name == "__init__.py":
        return True
    return False

filtered = [p for p in all_py if not is_excluded(p)]
excluded = [p for p in all_py if is_excluded(p)]

print(f"Tổng file .py        : {len(all_py)}")
print(f"Source sau khi lọc   : {len(filtered)}")
print(f"Đã loại (test/setup) : {len(excluded)}")


Tổng file .py        : 4496
Source sau khi lọc   : 2471
Đã loại (test/setup) : 2025


### Kết quả

| Chỉ số | Giá trị |
|---|---|
| Tổng file `.py` phát hiện | **4496** |
| Source sau khi lọc | **2471** |
| Đã loại (test/setup/init) | **2025** |

**Nhận xét:** Repo lớn bất thường vì `transformers-pr-agent` **bundle nguyên bộ thư viện `transformers`**
(thư mục `src` ~2779 file, `tests` ~1516 file). Con số 4496 là hợp lệ (đã kiểm tra không nhiễm venv).


### 3. Phân bố theo thư mục cấp 1

In [11]:
from collections import Counter
top = Counter(p.relative_to(REPO).parts[0] for p in all_py)
for d, n in top.most_common():
    print(f"{n:5d}  {d}")


 2779  src
 1516  tests
   96  examples
   75  utils
   11  docs
    6  benchmark_v2
    5  benchmark
    3  scripts
    2  .circleci
    1  conftest.py
    1  setup.py
    1  .github


### Reflection

- **Worked:** `pathlib.rglob` liệt kê nhanh, gọn; rule lọc đơn giản đủ dùng và không loại nhầm source thật.

- **Surprise / khó khăn:** repo hoá ra chứa nguyên `transformers`nên có 4496 file, quá lớn để chạy pipeline toàn bộ.
  Ban đầu nghi nhiễm virtualenv, nhưng kiểm tra phân bố thư mục cấp 1 xác nhận sạch. Phát hiện vài file test lọt lưới (chữ "test" nằm giữa tên như `modeling_test_detr.py`).

- **Resolved:** giữ rule lọc đơn giản + ghi chú minh bạch; quyết định **Task 2 chạy trên một tập con nhỏ** thay vì toàn bộ 4496 file (đúng tinh thần "chạy được ở quy mô nhỏ").
